In [5]:
import json
import requests
from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt


alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [6]:
df = pd.read_csv(
    '../dpt2020.csv',
    sep=';',
    dtype={'dpt': str, 'annais': str, 'sexe': int, 'nombre': int},
)

# Remove national aggregates (XX), year aggregates (XXXX), and rare-name bucket
df = df[
    (df['dpt'] != 'XX') &
    (df['annais'] != 'XXXX') &
    (df['preusuel'] != '_PRENOMS_RARES')
].copy()

df['annais'] = df['annais'].astype(int)
df['decade'] = (df['annais'] // 10) * 10

print(f"Rows after filter: {len(df):,}")
print(f"Departments: {df['dpt'].nunique()}")
print(f"Decades: {sorted(df['decade'].unique())}")

Rows after filter: 3,668,274
Departments: 99
Decades: [np.int64(1900), np.int64(1910), np.int64(1920), np.int64(1930), np.int64(1940), np.int64(1950), np.int64(1960), np.int64(1970), np.int64(1980), np.int64(1990), np.int64(2000), np.int64(2010), np.int64(2020)]


# Visualisation 1

In [ ]:
top_n = 15
names_by_year_gender = df.groupby(['annais', 'preusuel', 'sexe'], as_index=False)['nombre'].sum()

top_names_by_year_gender = (
    names_by_year_gender
    .sort_values(['annais', 'sexe', 'nombre'], ascending=[True, True, False])
    .groupby(['annais', 'sexe'], group_keys=False)
    .head(top_n)
    .reset_index(drop=True)
)

gender_map = {1: 'Boys', 2: 'Girls'}
top_names_by_year_gender['gender'] = top_names_by_year_gender['sexe'].map(gender_map)

top_names_by_year_gender['year'] = top_names_by_year_gender['annais'].astype(int)

max_births = top_names_by_year_gender.groupby(['year', 'sexe'])['nombre'].nlargest(top_n).groupby('year').max().max()

year_slider = alt.binding_range(
    min=int(top_names_by_year_gender['year'].min()),
    max=int(top_names_by_year_gender['year'].max()),
    step=1
)
year_selection = alt.param(
    name='Year',
    bind=year_slider,
    value=int(top_names_by_year_gender['year'].min())
)

# Create the chart
base = alt.Chart(top_names_by_year_gender).add_params(
    year_selection
).transform_filter(
    alt.datum.year == year_selection
).mark_bar()

boys_chart = base.transform_filter(
    alt.datum.sexe == 1
).encode(
    x=alt.X('nombre:Q', title='Number of births', scale=alt.Scale(reverse=True, domain=[0, max_births])),
    y=alt.Y('preusuel:N', sort='-x', title='Name'),
    color=alt.Color('gender:N', title='Gender', scale=alt.Scale(scheme='set2'))
).properties(width=300, height=600, title='Boys')

girls_chart = base.transform_filter(
    alt.datum.sexe == 2
).encode(
    x=alt.X('nombre:Q', title='Number of births', scale=alt.Scale(domain=[0, max_births])),
    y=alt.Y('preusuel:N', sort='-x', title='Name', axis=alt.Axis(orient='right')),
    color=alt.Color('gender:N', title='Gender', scale=alt.Scale(scheme='set2'))
).properties(width=300, height=600, title='Girls')

chart = alt.hconcat(boys_chart, girls_chart, spacing=0).resolve_scale(y='independent')
chart

alt.HConcatChart(...)

# Visualisation 2

In [ ]:
# Function to compare two unisex name variants across years
def compare_name_variants(name1, name2, gender1=1, gender2=2):
    """
    Compare two name variants (e.g., Camil vs Camille, Raphael vs Raphael)
    
    Args:
        name1: First name (e.g., 'Camil')
        name2: Second name (e.g., 'Camille')
        gender1: Gender for first name (1=Boys, 2=Girls)
        gender2: Gender for second name (1=Boys, 2=Girls)
    
    Returns:
        A combined chart showing both names' evolution over years
    """
    
    # Filter data for the two names
    data1 = names_by_year_gender[
        (names_by_year_gender['preusuel'].str.upper() == name1.upper()) &
        (names_by_year_gender['sexe'] == gender1)
    ].copy()
    
    data2 = names_by_year_gender[
        (names_by_year_gender['preusuel'].str.upper() == name2.upper()) &
        (names_by_year_gender['sexe'] == gender2)
    ].copy()
    
    if data1.empty and data2.empty:
        print(f"No data found for '{name1}' ({['Boys', 'Girls'][gender1-1]}) or '{name2}' ({['Boys', 'Girls'][gender2-1]})")
        return None
    
    # Add labels for clarity
    data1['pair'] = f"{data1['preusuel'].iloc[0]} ({['Boys', 'Girls'][gender1-1]})" if not data1.empty else f"{name1} ({['Boys', 'Girls'][gender1-1]})"
    data2['pair'] = f"{data2['preusuel'].iloc[0]} ({['Boys', 'Girls'][gender2-1]})" if not data2.empty else f"{name2} ({['Boys', 'Girls'][gender2-1]})"
    
    # Combine data
    combined_data = pd.concat([data1, data2], ignore_index=True)
    combined_data['year'] = combined_data['annais'].astype(int)
    
    # Create line chart showing evolution over years
    chart = alt.Chart(combined_data).mark_line(point=True, size=3).encode(
        x=alt.X('year:O', title='Year'),
        y=alt.Y('nombre:Q', title='Number of births'),
        color=alt.Color('pair:N', title='Name (Gender)', scale=alt.Scale(scheme='set2')),
        strokeDash=alt.StrokeDash('pair:N', legend=None),
        tooltip=['year:O', 'nombre:Q', 'pair:N']
    ).properties(
        width=800,
        height=400,
        title={
            "text": f"Evolution of '{name1}' and '{name2}' across years",
            "fontSize": 14
        }
    ).interactive()
    
    return chart

chart = compare_name_variants('Claude', 'Claude', gender1=1, gender2=2)
chart

# Visualisation 3

In [ ]:
# Function to compare multiple name pairs with combined pie charts and year slider
def compare_name_pairs_pie(name_pairs_list):
    """
    Compare multiple name pairs with combined pie charts (one per pair) and year slider
    Shows both boys and girls even if missing data (fills with 0)
    
    Args:
        name_pairs_list: List of tuples like [
            ('Camille', 1, 'Camille', 2),
            ('Raphael', 1, 'Raphael', 2),
        ]
        where each tuple is (name1, gender1, name2, gender2)
    
    Returns:
        A combined visualization with pie charts and year slider
    """
    
    all_pie_data = []
    
    # Get all years from the data
    all_years = names_by_year_gender['annais'].unique()
    
    # Collect data for all name pairs
    for name1, gender1, name2, gender2 in name_pairs_list:
        data1 = names_by_year_gender[
            (names_by_year_gender['preusuel'].str.upper() == name1.upper()) &
            (names_by_year_gender['sexe'] == gender1)
        ].copy()
        
        data2 = names_by_year_gender[
            (names_by_year_gender['preusuel'].str.upper() == name2.upper()) &
            (names_by_year_gender['sexe'] == gender2)
        ].copy()
        
        # Create pair identifier
        pair_id = f"{name1} vs {name2}"
        
        # Prepare data1
        if not data1.empty:
            data1['pair'] = pair_id
            data1['display_name'] = f"{data1['preusuel'].iloc[0]} ({['Boys', 'Girls'][gender1-1]})"
            data1_years = set(data1['annais'].unique())
        else:
            data1_years = set()
        
        # Prepare data2
        if not data2.empty:
            data2['pair'] = pair_id
            data2['display_name'] = f"{data2['preusuel'].iloc[0]} ({['Boys', 'Girls'][gender2-1]})"
            data2_years = set(data2['annais'].unique())
        else:
            data2_years = set()
        
        # Find missing years for data1
        missing_years_1 = all_years[~np.isin(all_years, list(data1_years))]
        if len(missing_years_1) > 0 and not data1.empty:
            # Add rows with 0 for missing years
            for year in missing_years_1:
                data1 = pd.concat([data1, pd.DataFrame({
                    'annais': [year],
                    'preusuel': [data1['preusuel'].iloc[0]],
                    'sexe': [gender1],
                    'nombre': [0],
                    'pair': [pair_id],
                    'display_name': [f"{data1['preusuel'].iloc[0]} ({['Boys', 'Girls'][gender1-1]})"]
                })], ignore_index=True)
        
        # Find missing years for data2
        missing_years_2 = all_years[~np.isin(all_years, list(data2_years))]
        if len(missing_years_2) > 0 and not data2.empty:
            # Add rows with 0 for missing years
            for year in missing_years_2:
                data2 = pd.concat([data2, pd.DataFrame({
                    'annais': [year],
                    'preusuel': [data2['preusuel'].iloc[0]],
                    'sexe': [gender2],
                    'nombre': [0],
                    'pair': [pair_id],
                    'display_name': [f"{data2['preusuel'].iloc[0]} ({['Boys', 'Girls'][gender2-1]})"]
                })], ignore_index=True)
        
        # Also add empty variants that don't exist at all
        if data1.empty:
            for year in all_years:
                data1_temp = pd.DataFrame({
                    'annais': [year],
                    'preusuel': [name1],
                    'sexe': [gender1],
                    'nombre': [0],
                    'pair': [pair_id],
                    'display_name': [f"{name1} ({['Boys', 'Girls'][gender1-1]})"]
                })
                all_pie_data.append(data1_temp)
        else:
            all_pie_data.append(data1)
        
        if data2.empty:
            for year in all_years:
                data2_temp = pd.DataFrame({
                    'annais': [year],
                    'preusuel': [name2],
                    'sexe': [gender2],
                    'nombre': [0],
                    'pair': [pair_id],
                    'display_name': [f"{name2} ({['Boys', 'Girls'][gender2-1]})"]
                })
                all_pie_data.append(data2_temp)
        else:
            all_pie_data.append(data2)
    
    if not all_pie_data:
        print("No data found for any of the name pairs")
        return None
    
    # Combine all data
    all_data = pd.concat(all_pie_data, ignore_index=True)
    all_data['year'] = all_data['annais'].astype(int)
    
    # Create year slider
    year_slider = alt.binding_range(
        min=int(all_data['year'].min()),
        max=int(all_data['year'].max()),
        step=1
    )
    year_selection = alt.param(
        name='Year',
        bind=year_slider,
        value=int(all_data['year'].min())
    )
    
    # Create pie charts - one per pair, showing both boys and girls distribution
    pie_chart = alt.Chart(all_data).add_params(
        year_selection
    ).transform_filter(
        alt.datum.year == year_selection
    ).mark_arc(innerRadius=0).encode(
        theta=alt.Theta('nombre:Q'),
        color=alt.Color('display_name:N', title='Name (Gender)', scale=alt.Scale(scheme='set2')),
        tooltip=['display_name:N', 'nombre:Q', 'year:O']
    ).properties(
        width=200,
        height=200
    ).facet(
        column=alt.Column('pair:N', header=alt.Header(titleFontSize=12, labelFontSize=10))
    ).resolve_scale(
        color='independent'
    ).properties(
        title={
            "text": "Name Pair Distribution by Year (Boys vs Girls)",
            "subtitle": "Each pie shows the distribution between the two name variants. Use slider to change year.",
            "fontSize": 14
        }
    )
    
    return pie_chart

# Example usage with multiple name pairs
name_pairs_pie = [
    ('Camille', 1, 'Camille', 2),
    ('Claude', 1, 'Claude', 2),
    ('Dominique', 1, 'Dominique', 2)
]

chart_pie = compare_name_pairs_pie(name_pairs_pie)
chart_pie